# **Resources**

- https://pysal.org/scipy2019-intermediate-gds/deterministic/gds2-rasters.html

In [ ]:
import numpy as np
import pandas as pd

import rasterio as rio
from rasterio.mask import mask
from rasterstats import zonal_stats

import geopandas as gpd

import matplotlib.pyplot as plt

# **TRACTS**

In [ ]:
tracts = gpd.read_file('https://services5.arcgis.com/GfwWNkhOj9bNBqoJ/arcgis/rest/services/NYC_Census_Tracts_for_2010_US_Census/FeatureServer/0/query?where=1=1&outFields=*&outSR=4326&f=pgeojson')
tracts = tracts.to_crs('EPSG:4326')
tracts = tracts[tracts.BoroName == 'Brooklyn']
tracts = tracts.drop(columns=['OBJECTID', 'CTLabel', 'BoroCode', 'BoroName', 'BoroCT2010', 'CDEligibil', 'NTACode', 'NTAName', 'PUMA', 'Shape__Area', 'Shape__Length'])
tracts = tracts.reset_index(drop=True)

In [ ]:
tracts_bbox = tracts.total_bounds

In [ ]:
lst_file = rio.open('data/rasters/lst.tiff')
lst_window = lst_file.window(*tracts_bbox)
lst_bk = lst_file.read(1, window=lst_window)

plt.figure(figsize=(15,15));
plt.imshow(lst_bk, extent=tracts_bbox[[0,2,1,3]], cmap='hot');

In [ ]:
ndvi_file = rio.open('data/rasters/ndvi.tif')
ndvi_window = ndvi_file.window(*tracts_bbox)
ndvi_bk = ndvi_file.read(1, window=ndvi_window)

plt.figure(figsize=(15,15));
plt.imshow(ndvi_bk, extent=tracts_bbox[[0, 2, 1, 3]], cmap='Greens');

In [ ]:
albedo_file = rio.open('data/rasters/albedo.tif')
albedo_window = albedo_file.window(*tracts_bbox)
albedo_bk = albedo_file.read(1, window=albedo_window)

plt.figure(figsize=(15,15));
plt.imshow(albedo_bk, extent=tracts_bbox[[0,2,1,3]], cmap="Greys");

In [ ]:
def clean_mask(geom, dataset=lst_file, **mask_kw):
    mask_kw.setdefault('crop', True)
    mask_kw.setdefault('all_touched', True)
    mask_kw.setdefault('filled', False)
    masked, mask_transform = mask(dataset=dataset, shapes=(geom,), **mask_kw)
    return masked



lsts = []
for i in range(len(tracts)):
    try:
        lst = tracts.geometry[[i]].apply(clean_mask).apply(np.ma.median).values[0]
    except:
        lst = np.nan
    
    lsts.append(lst)
    
    
tracts['lst'] = lsts

In [ ]:
def clean_mask(geom, dataset=ndvi_file, **mask_kw):
    mask_kw.setdefault('crop', True)
    mask_kw.setdefault('all_touched', True)
    mask_kw.setdefault('filled', False)
    masked, mask_transform = mask(dataset=dataset, shapes=(geom,), **mask_kw)
    return masked



ndvis = []
for i in range(len(tracts)):
    try:
        ndvi = tracts.geometry[[i]].apply(clean_mask).apply(np.ma.median).values[0]
    except:
        ndvi = np.nan
    
    ndvis.append(ndvi)
    
    
tracts['ndvi'] = ndvis

In [ ]:
def clean_mask(geom, dataset=albedo_file, **mask_kw):
    mask_kw.setdefault('crop', True)
    mask_kw.setdefault('all_touched', True)
    mask_kw.setdefault('filled', False)
    masked, mask_transform = mask(dataset=dataset, shapes=(geom,), **mask_kw)
    return masked


albedos = []
for i in range(len(tracts)):
    try:
        albedo = tracts.geometry[[i]].apply(clean_mask).apply(np.ma.median).values[0]
    except:
        albedo = np.nan
    
    albedos.append(albedo)
    
    
tracts['albedo'] = albedos

In [ ]:
tracts.to_file('data/tracts_with_raster_values.shp')

# **NTAs**

In [ ]:
nta = gpd.read_file('https://services5.arcgis.com/GfwWNkhOj9bNBqoJ/arcgis/rest/services/NYC_Neighborhood_Tabulation_Areas_2020/FeatureServer/0/query?where=1=1&outFields=*&outSR=4326&f=pgeojson')
nta = nta.to_crs('EPSG:4326')
nta = nta[nta.BoroName == 'Brooklyn']
nta = nta.drop(columns=['OBJECTID', 'BoroCode', 'CountyFIPS', 'NTAAbbrev', 'NTAType', 'CDTA2020', 'CDTAName', 'Shape__Area', 'Shape__Length', 'BoroName']) 
nta = nta.reset_index(drop=True)

In [ ]:
nta_bbox = nta.total_bounds

In [ ]:
lst_file = rio.open('data/rasters/lst.tiff')
lst_window = lst_file.window(*nta_bbox)
lst_bk = lst_file.read(1, window=lst_window)

plt.figure(figsize=(15,15));
plt.imshow(lst_bk, extent=nta_bbox[[0,2,1,3]], cmap='hot');

In [ ]:
ndvi_file = rio.open('data/rasters/ndvi.tif')
ndvi_window = ndvi_file.window(*nta_bbox)
ndvi_bk = ndvi_file.read(1, window=ndvi_window)

plt.figure(figsize=(15,15));
plt.imshow(ndvi_bk, extent=nta_bbox[[0, 2, 1, 3]], cmap='Greens');

In [ ]:
albedo_file = rio.open('data/rasters/albedo.tif')
albedo_window = albedo_file.window(*nta_bbox)
albedo_bk = albedo_file.read(1, window=albedo_window)

plt.figure(figsize=(15,15));
plt.imshow(albedo_bk, extent=nta_bbox[[0,2,1,3]], cmap="Greys");

In [ ]:
def clean_mask(geom, dataset=lst_file, **mask_kw):
    mask_kw.setdefault('crop', True)
    mask_kw.setdefault('all_touched', True)
    mask_kw.setdefault('filled', False)
    masked, mask_transform = mask(dataset=dataset, shapes=(geom,), **mask_kw)
    return masked



lsts = []
for i in range(len(nta)):
    try:
        lst = nta.geometry[[i]].apply(clean_mask).apply(np.ma.median).values[0]
    except:
        lst = np.nan
    
    lsts.append(lst)
    
    
nta['lst'] = lsts

In [ ]:
def clean_mask(geom, dataset=ndvi_file, **mask_kw):
    mask_kw.setdefault('crop', True)
    mask_kw.setdefault('all_touched', True)
    mask_kw.setdefault('filled', False)
    masked, mask_transform = mask(dataset=dataset, shapes=(geom,), **mask_kw)
    return masked



ndvis = []
for i in range(len(nta)):
    try:
        ndvi = nta.geometry[[i]].apply(clean_mask).apply(np.ma.median).values[0]
    except:
        ndvi = np.nan
    
    ndvis.append(ndvi)
    
    
nta['ndvi'] = ndvis

In [ ]:
def clean_mask(geom, dataset=albedo_file, **mask_kw):
    mask_kw.setdefault('crop', True)
    mask_kw.setdefault('all_touched', True)
    mask_kw.setdefault('filled', False)
    masked, mask_transform = mask(dataset=dataset, shapes=(geom,), **mask_kw)
    return masked


albedos = []
for i in range(len(nta)):
    try:
        albedo = nta.geometry[[i]].apply(clean_mask).apply(np.ma.median).values[0]
    except:
        albedo = np.nan
    
    albedos.append(albedo)
    
    
nta['albedo'] = albedos

# **BUILDINGS**

In [ ]:
buildings = gpd.read_file('data/buildings_bk.shp')

buildings = buildings.set_crs('EPSG:26918')
buildings.plot(figsize=(20,20));

In [ ]:
buildings_with_buffer = gpd.GeoDataFrame(geometry=buildings.buffer(15))
buildings_with_buffer = buildings_with_buffer.to_crs('EPSG:4326')
buildings = buildings.to_crs('EPSG:4326')

buildings_with_buffer.plot(figsize=(20,20));

In [ ]:
buildings = buildings.drop(columns=['height', 'nID', 'bID'])
buildings_with_buffer['uID'] = buildings['uID']

In [ ]:
buildings_bbox = buildings.total_bounds

In [ ]:
lst_file = rio.open('data/rasters/lst.tiff')
lst_window = lst_file.window(*buildings_bbox)
lst_bk = lst_file.read(1, window=lst_window)

plt.figure(figsize=(15,15));
plt.imshow(lst_bk, extent=buildings_bbox[[0,2,1,3]], cmap='hot');

In [ ]:
ndvi_file = rio.open('data/rasters/ndvi.tif')
ndvi_window = ndvi_file.window(*buildings_bbox)
ndvi_bk = ndvi_file.read(1, window=ndvi_window)

plt.figure(figsize=(15,15));
plt.imshow(ndvi_bk, extent=buildings_bbox[[0, 2, 1, 3]], cmap='Greens');

In [ ]:
albedo_file = rio.open('data/rasters/albedo.tif')
albedo_window = albedo_file.window(*buildings_bbox)
albedo_bk = albedo_file.read(1, window=albedo_window)

plt.figure(figsize=(15,15));
plt.imshow(albedo_bk, extent=buildings_bbox[[0,2,1,3]], cmap="Greys");

In [ ]:
def clean_mask(geom, dataset=lst_file, **mask_kw):
    mask_kw.setdefault('crop', True)
    mask_kw.setdefault('all_touched', True)
    mask_kw.setdefault('filled', False)
    masked, mask_transform = mask(dataset=dataset, shapes=(geom,), **mask_kw)
    return masked



lsts = []
for i in range(len(buildings)):
    try:
        lst = buildings.geometry[[i]].apply(clean_mask).apply(np.ma.median).values[0]
    except:
        lst = np.nan
    
    lsts.append(lst)
    
    
buildings['lst'] = lsts

In [ ]:
def clean_mask(geom, dataset=ndvi_file, **mask_kw):
    mask_kw.setdefault('crop', True)
    mask_kw.setdefault('all_touched', True)
    mask_kw.setdefault('filled', False)
    masked, mask_transform = mask(dataset=dataset, shapes=(geom,), **mask_kw)
    return masked



ndvis = []
for i in range(len(buildings)):
    try:
        ndvi = buildings_with_buffer.geometry[[i]].apply(clean_mask).apply(np.ma.median).values[0]
    except:
        ndvi = np.nan
    
    ndvis.append(ndvi)
    
    
buildings['ndvi'] = ndvis

In [ ]:
def clean_mask(geom, dataset=albedo_file, **mask_kw):
    mask_kw.setdefault('crop', True)
    mask_kw.setdefault('all_touched', True)
    mask_kw.setdefault('filled', False)
    masked, mask_transform = mask(dataset=dataset, shapes=(geom,), **mask_kw)
    return masked


albedos = []
for i in range(len(buildings)):
    try:
        albedo = buildings.geometry[[i]].apply(clean_mask).apply(np.ma.median).values[0]
    except:
        albedo = np.nan
    
    albedos.append(albedo)
    
    
buildings['albedo'] = albedos

In [ ]:
gpd.GeoDataFrame(buildings).to_file('data/brooklyn_buildings_with_raster_values.shp')